In [50]:
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/whoop_fitness_dataset_100k.csv")


In [51]:
def drop_irr_features(df):  
    col_to_drop = [
        "user_id",
        "date",
        "day_of_week",
        "primary_sport",
        "workout_time_of_day"
    ]
    df1 = df.copy();
    df1.drop(columns=col_to_drop, inplace=True)
    return df1

In [ ]:

important_metrics = [
    "hrv",
    "resting_heart_rate",
    "respiratory_rate",
    "skin_temp_deviation",
    "sleep_hours",
    "sleep_efficiency",
    "rem_sleep_hours",
    "deep_sleep_hours",
    "light_sleep_hours",
    "wake_ups",
    "day_strain",
    "activity_strain",
    "activity_duration_min",
]


def historical_feature_generation(df):
    df1 = df.copy()
    df1 = df1.sort_values(["user_id", "date"]).copy()
    
    df1["high_intensity_min"] = df1["hr_zone_3_min"] + df1["hr_zone_4_min"] + df1["hr_zone_5_min"]
    df1["low_intensity_min"] = df1["hr_zone_1_min"] + df1["hr_zone_2_min"]
    df1["hrv_ratio"] = df1["hrv"] / df1["hrv_baseline"]
    df1["rhr_ratio"] = df1["resting_heart_rate"] / df1["rhr_baseline"]
    df1["rem_sleep_hours_ratio"] = df1["rem_sleep_hours"] / df1["sleep_hours"]
    df1["deep_sleep_hours_ratio"] = df1["deep_sleep_hours"] / df1["sleep_hours"]
    df1["bmi"] = df1["weight_kg"] / np.exp2(df1["height_cm"] / 100)
    
    for col in important_metrics:
        df1[f"previous_{col}"] = (
            df1.groupby("user_id")[col].shift(1)
        )

    for col in important_metrics:
        df1[f"{col}_change"] = (
            df1[col] - df1[f"previous_{col}"]
        )
        
    df1["is_first_record"] = df1["previous_hrv"].isna().astype(int)

    return df1

In [57]:
def daily_feature_generation(df):
    df1 = df.copy()
    
    df1["high_intensity_min"] = df1["hr_zone_3_min"] + df1["hr_zone_4_min"] + df1["hr_zone_5_min"]
    df1["low_intensity_min"] = df1["hr_zone_1_min"] + df1["hr_zone_2_min"]
    df1["hrv_ratio"] = df1["hrv"] / df1["hrv_baseline"]
    df1["rhr_ratio"] = df1["resting_heart_rate"] / df1["rhr_baseline"]
    df1["rem_sleep_hours_ratio"] = df1["rem_sleep_hours"] / df1["sleep_hours"]
    df1["deep_sleep_hours_ratio"] = df1["deep_sleep_hours"] / df1["sleep_hours"]
    df1["bmi"] = df1["weight_kg"] / (df1["height_cm"] / 100) ** 2

    return df1

In [53]:
def create_temp_strat_bin(df):
    df1 = df.copy()
    df1['score_bins'] = pd.qcut(df['recovery_score'], q=10, labels=False, duplicates='drop')
    return df1

In [54]:
print("functions.py imported successfully")

functions.py imported successfully
